# 03 - Exploratory Data Analysis
**Objective:** describe customers and explore how behavior relates to the Unpaid Tagging indicator.
**Terms:** *tagged* = 1, *non-tagged* = 0 (definition not provided; not assumed to be default).
**Rules:** every rate is shown with its group size; all relationships are *associations*, not causes.
Data All is analysed with its Q1-Q4 label, Data Validasi with its Q2-Q5 label. Outputs are cleared before committing.

In [ ]:
import sys
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from src.preprocessing import load_clean, TARGET, ID_COL
from src.feature_engineering import build_features, WINDOW_ALL, WINDOW_VALIDASI
from src.eda import (rate_table, rate_panel, plot_rate_bars, savefig, cramers_v, PALETTE,
                     age_group, income_group, product_group, fit_balance_cutpoints,
                     balance_tier, assign_segments)

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)
sns.set_style("whitegrid"); plt.rcParams["figure.dpi"] = 100
FIG = "../figures"

clean_all = load_clean("../data/raw/finanku_data_all.csv")
clean_val = load_clean("../data/raw/finanku_data_validasi.csv")
FA = build_features(clean_all, WINDOW_ALL)
FV = build_features(clean_val, WINDOW_VALIDASI)
yA = clean_all.set_index(ID_COL)[TARGET]
yV = clean_val.set_index(ID_COL)[TARGET]
pA, pV = yA.mean(), yV.mean()
print(f"Tagging rate  All: {pA:.2%} | Validasi: {pV:.2%}")

## A. Overall customer profile
Profile columns are identical in both files, so Data All is used here.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(FA["age"], bins=20, color=PALETTE["All"], edgecolor="white")
ax[0].set_title("Age distribution", fontweight="bold"); ax[0].set_xlabel("Age"); ax[0].set_ylabel("Customers")
inc = FA.loc[FA["income_measure"] > 0, "income_measure"] / 1e6
ax[1].hist(inc, bins=25, color=PALETTE["All"], edgecolor="white")
ax[1].set_title(f"Income measure distribution (excl. {int((FA['income_measure']==0).sum())} zero values)", fontweight="bold")
ax[1].set_xlabel("Income measure (millions, unit unconfirmed)")
fig.tight_layout(); savefig(fig, "01_age_income_distribution.png", FIG); plt.show()
print(FA[["age", "income_measure"]].describe().round(0))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 2.2]})
city_n = FA["city"].value_counts()
ax[0].bar(city_n.index, city_n.values, color=PALETTE["All"])
for i, v in enumerate(city_n.values): ax[0].text(i, v, f"{v:,}\n({v/len(FA):.0%})", ha="center", va="bottom", fontsize=8)
ax[0].set_title("Customers by city", fontweight="bold"); ax[0].set_ylim(0, city_n.max() * 1.2)
br = FA.groupby(["city", "branch_code"]).size().reset_index(name="n").sort_values(["city", "branch_code"])
cols = {c: col for c, col in zip(city_n.index, ["#2E5E8C", "#6FA8DC", "#9DB8D2"])}
ax[1].bar(br["branch_code"], br["n"], color=br["city"].map(cols))
ax[1].set_title("Customers by branch (colour = city)", fontweight="bold"); ax[1].tick_params(axis="x", rotation=60, labelsize=8)
ax[1].legend(handles=[plt.Rectangle((0,0),1,1,color=c) for c in cols.values()], labels=list(cols.keys()), fontsize=8)
fig.tight_layout(); savefig(fig, "02_city_branch_distribution.png", FIG); plt.show()

print("Customers per branch (min / median / max):", br["n"].min(), br["n"].median(), br["n"].max())
print("Branches per city:\n", br.groupby("city")["branch_code"].nunique())
print("\nAge group share by city:")
display(pd.crosstab(FA["city"], age_group(FA["age"]), normalize="index").round(3))
print("Income group share by city:")
display(pd.crosstab(FA["city"], income_group(FA["income_measure"]), normalize="index").round(3))

## B. Unpaid Tagging analysis
Group size (`n`) and a Wilson 95% confidence interval accompany every rate. A branch with ~200 customers has a wide interval; differences inside the interval should not be over-read.

In [ ]:
ov = pd.DataFrame({"n": [len(yA), len(yV)], "tagged": [yA.sum(), yV.sum()]}, index=["All (Q1-Q4)", "Validasi (Q2-Q5)"])
ov["rate"] = ov["tagged"] / ov["n"]
from src.eda import wilson_ci
ov["ci_low"], ov["ci_high"] = wilson_ci(ov["tagged"], ov["n"])
display(ov.round(4))

# City and branch
t_city_a, t_city_v = rate_table(yA, FA["city"]), rate_table(yV, FV["city"])
rate_panel(t_city_a, t_city_v, pA, pV, "Tagging rate by city", "City", "03a_tag_rate_city.png", fig_dir=FIG); plt.show()
display(t_city_a.round(3)); display(t_city_v.round(3))

t_br_a, t_br_v = rate_table(yA, FA["branch_code"]), rate_table(yV, FV["branch_code"])
rate_panel(t_br_a, t_br_v, pA, pV, "Tagging rate by branch (whiskers = 95% CI)", "Branch code", "03b_tag_rate_branch.png",
           rotate=60, fig_dir=FIG, figsize=(12, 4)); plt.show()
print("Branch rate range  All:", f"{t_br_a['rate'].min():.1%}-{t_br_a['rate'].max():.1%}",
      "| Validasi:", f"{t_br_v['rate'].min():.1%}-{t_br_v['rate'].max():.1%}")
print("Branches whose 95% CI excludes the overall rate  All:", int(((t_br_a.ci_low > pA) | (t_br_a.ci_high < pA)).sum()),
      "| Validasi:", int(((t_br_v.ci_low > pV) | (t_br_v.ci_high < pV)).sum()), "of", len(t_br_a))

In [ ]:
t_age_a, t_age_v = rate_table(yA, age_group(FA["age"])), rate_table(yV, age_group(FV["age"]))
rate_panel(t_age_a, t_age_v, pA, pV, "Tagging rate by age group", "Age group", "04a_tag_rate_age.png", fig_dir=FIG); plt.show()
display(t_age_a.round(3)); display(t_age_v.round(3))

t_inc_a, t_inc_v = rate_table(yA, income_group(FA["income_measure"])), rate_table(yV, income_group(FV["income_measure"]))
rate_panel(t_inc_a, t_inc_v, pA, pV, "Tagging rate by income group (unit unconfirmed)", "Income measure band", "04b_tag_rate_income.png", fig_dir=FIG); plt.show()
display(t_inc_a.round(3)); display(t_inc_v.round(3))

## C. Quarterly behavioral analysis
Q2-Q4 are identical in both files; Q5 exists only in Data Validasi (shown separately).

In [ ]:
def quarter_stats(df, q):
    return {"quarter": q.upper(),
            "mean_balance_M": df[f"balance_{q}"].mean() / 1e6,
            "zero_balance_pct": (df[f"balance_{q}"] == 0).mean(),
            "mean_products": df[f"num_products_{q}"].mean(),
            "active_pct": df[f"active_member_{q}"].mean(),
            "card_pct": df[f"has_credit_card_{q}"].mean()}
qs = pd.DataFrame([quarter_stats(clean_all, q) for q in ["q1", "q2", "q3", "q4"]] + [quarter_stats(clean_val, "q5")]).set_index("quarter")
display(qs.round(4))

fig, ax = plt.subplots(1, 5, figsize=(15, 3.2))
spec = [("mean_balance_M", "Mean balance (millions)"), ("zero_balance_pct", "Zero-balance customers"),
        ("mean_products", "Mean products"), ("active_pct", "Active-member rate"), ("card_pct", "Credit-card ownership")]
for a, (col, ttl) in zip(ax, spec):
    a.plot(qs.index[:4], qs[col][:4], marker="o", color=PALETTE["All"])
    a.plot(qs.index[3:], qs[col][3:], marker="o", ls="--", color=PALETTE["Validasi"])
    a.set_title(ttl, fontsize=9, fontweight="bold")
    if col.endswith("pct"): a.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.suptitle("Quarterly behaviour (orange dashed segment = Q4 to Q5, Data Validasi only)", y=1.04, fontweight="bold")
fig.tight_layout(); savefig(fig, "05_quarterly_behaviour.png", FIG); plt.show()

In [ ]:
def transitions(df, col, q0, q1):
    d = df[f"{col}_{q1}"] - df[f"{col}_{q0}"]
    return {"up": int((d > 0).sum()), "down": int((d < 0).sum()), "same": int((d == 0).sum())}

rows = []
for src_name, df, pairs in [("All", clean_all, [("q1","q2"),("q2","q3"),("q3","q4")]), ("Validasi", clean_val, [("q4","q5")])]:
    for q0, q1 in pairs:
        for col in ["balance", "num_products", "active_member"]:
            rows.append({"step": f"{q0.upper()}->{q1.upper()}", "metric": col, **transitions(df, col, q0, q1)})
display(pd.DataFrame(rows).pivot(index="step", columns="metric", values=["up", "down", "same"]))

## D. Relationship analysis
Each chart compares tagging rates across behaviour groups in both windows (group sizes shown inside bars).

In [ ]:
cuts = fit_balance_cutpoints(FA)          # fitted on Data All, applied to both windows
print("Balance tier cut points (tertiles of non-zero customers, Data All):", [f"{c/1e6:.1f}M" for c in cuts])
FA["balance_tier"], FV["balance_tier"] = balance_tier(FA, cuts), balance_tier(FV, cuts)

def panel(label, fa, fv, fname, **kw):
    ta, tv = rate_table(yA, fa), rate_table(yV, fv)
    rate_panel(ta, tv, pA, pV, f"Tagging rate by {label}", label, fname, fig_dir=FIG, **kw); plt.show()
    return ta, tv

ta, tv = panel("number of products (end of window)", FA["products_end"], FV["products_end"], "06a_tag_rate_products.png")
display(ta.round(3)); display(tv.round(3))

In [ ]:
ta, tv = panel("balance tier (mean balance)", FA["balance_tier"], FV["balance_tier"], "06b_tag_rate_balance_tier.png")
display(ta.round(3)); display(tv.round(3))
ta, tv = panel("zero-balance quarters (0-4)", FA["n_zero_balance_quarters"], FV["n_zero_balance_quarters"], "06c_tag_rate_zero_quarters.png")
ta, tv = panel("active quarters (0-4)", FA["n_active_quarters"], FV["n_active_quarters"], "06d_tag_rate_active_quarters.png")
display(ta.round(3)); display(tv.round(3))

### Credit-card status (important)
In Data All the card indicator is identical for Q1-Q3 and equals 1 for everyone in Q4. Customers without a card up to Q3 are examined separately.

In [ ]:
lab = lambda s: s.map({0: "No card until final quarter", 1: "Card throughout"})
ta, tv = panel("credit-card status", lab(FA["card_start"]), lab(FV["card_start"]), "07_tag_rate_card_status.png")
display(ta.round(3)); display(tv.round(3))

lab_tab = pd.DataFrame({"no_card_group": FA["card_start"] == 0, "tag_all": yA, "tag_validasi": yV})
print("\nLabel transitions inside the no-card group (All -> Validasi):")
display(pd.crosstab(lab_tab.loc[lab_tab.no_card_group, "tag_all"], lab_tab.loc[lab_tab.no_card_group, "tag_validasi"]))
print("Share of all tagged customers that are in the no-card group  All:",
      f"{yA[FA['card_start']==0].sum()/yA.sum():.1%}", "| Validasi:", f"{yV[FV['card_start']==0].sum()/yV.sum():.1%}")

In [ ]:
# Numeric features: tagged vs non-tagged (medians) - descriptive only
cols = ["age", "income_measure", "balance_mean", "products_mean", "active_rate", "balance_change_abs"]
for name, F, y in [("All", FA, yA), ("Validasi", FV, yV)]:
    print(f"--- {name}: median by tag status ---")
    display(F[cols].groupby(y.values).median().rename(index={0: "non-tagged", 1: "tagged"}).round(2))

## E. Correlation analysis
Spearman rank correlation (robust to skew, works for binary/ordinal variables) between each reduced-set feature and the tag, and Cramer's V for categorical variables.
Redundant features (see data dictionary) are excluded. **Correlation does not imply causation.**

In [ ]:
feat = ["age", "income_measure", "income_is_zero", "balance_mean", "balance_std", "balance_change_abs",
        "n_zero_balance_quarters", "products_mean", "products_max", "products_change",
        "active_rate", "active_change", "card_start"]
def spearman_with_target(F, y):
    return pd.Series({c: spearmanr(F[c], y.loc[F.index])[0] for c in feat})
sp = pd.DataFrame({"All": spearman_with_target(FA, yA), "Validasi": spearman_with_target(FV, yV)}).sort_values("All")
display(sp.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = np.arange(len(sp)); h = 0.38
ax.barh(y_pos - h/2, sp["All"], h, color=PALETTE["All"], label="Data All (Q1-Q4)")
ax.barh(y_pos + h/2, sp["Validasi"], h, color=PALETTE["Validasi"], label="Data Validasi (Q2-Q5)")
ax.set_yticks(y_pos); ax.set_yticklabels(sp.index, fontsize=9); ax.axvline(0, color="#333", lw=0.8)
ax.set_title("Spearman correlation with tagging indicator", fontweight="bold"); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "08a_spearman_with_tag.png", FIG); plt.show()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(FA[feat].corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", center=0, annot_kws={"size": 7}, ax=ax)
ax.set_title("Spearman correlation among features (Data All)", fontweight="bold")
fig.tight_layout(); savefig(fig, "08b_feature_correlation_heatmap.png", FIG); plt.show()

print("Cramer's V with tag (0 = none, 1 = perfect):")
for c in ["city", "branch_code"]:
    print(f"  {c:12s} All: {cramers_v(FA[c], yA):.3f} | Validasi: {cramers_v(FV[c], yV):.3f}")

## F. Customer segmentation (rule-based)
Rules are transparent and identical in both windows: credit-card status first, then product count x whether the customer has any balance. Business rules were chosen over clustering because the signal is concentrated in a few discrete variables, and rules are easy to explain and monitor.

In [ ]:
FA["segment"], FV["segment"] = assign_segments(FA), assign_segments(FV)
sa, sv = rate_table(yA, FA["segment"]), rate_table(yV, FV["segment"])
order = sa.sort_values("rate", ascending=False).index
seg = pd.concat({"All": sa.loc[order, ["n", "rate", "lift_vs_overall", "share_of_tagged"]],
                 "Validasi": sv.loc[order, ["n", "rate", "lift_vs_overall", "share_of_tagged"]]}, axis=1)
display(seg.round(3))

fig, ax = plt.subplots(figsize=(10, 4.2)); x = np.arange(len(order)); w = 0.38
ax.bar(x - w/2, sa.loc[order, "rate"], w, color=PALETTE["All"], label="Data All (Q1-Q4)")
ax.bar(x + w/2, sv.loc[order, "rate"], w, color=PALETTE["Validasi"], label="Data Validasi (Q2-Q5)")
for i, s in enumerate(order):
    ax.text(i - w/2, sa.loc[s, "rate"] + 0.01, f"n={sa.loc[s,'n']:,}", ha="center", fontsize=6.5)
    ax.text(i + w/2, sv.loc[s, "rate"] + 0.01, f"n={sv.loc[s,'n']:,}", ha="center", fontsize=6.5)
ax.axhline(pA, ls="--", lw=0.8, color=PALETTE["All"]); ax.axhline(pV, ls="--", lw=0.8, color=PALETTE["Validasi"])
ax.set_xticks(x); ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.set_title("Tagging rate by rule-based segment", fontweight="bold"); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "09_segmentation.png", FIG); plt.show()

In [ ]:
# Is segmentation actionable? Spread, concentration and stability across windows
for name, s, y in [("All", sa, yA), ("Validasi", sv, yV)]:
    print(f"{name}: rate spread across segments = {s['rate'].min():.1%} to {s['rate'].max():.1%}")
top2 = sa.sort_values("rate", ascending=False).head(2)
print("\nTop-2 segments in Data All:", list(top2.index),
      f"-> {top2['share_of_customers'].sum():.1%} of customers, {top2['share_of_tagged'].sum():.1%} of tagged customers")
rho = spearmanr(sa["rate"], sv.loc[sa.index, "rate"])[0]
print(f"Rank agreement of segment rates between windows (Spearman): {rho:.2f}")

## G. Sensitivity check: card holders only
The no-card group (100% tagged in Data All) dominates the overall pattern and its meaning is unknown. Here the main relationships are re-checked **without** that group to see whether other signals survive.

In [ ]:
m_a, m_v = FA["card_start"] == 1, FV["card_start"] == 1
yA_c, yV_c = yA[m_a], yV[m_v]
print(f"Card holders: All n={m_a.sum():,}, rate={yA_c.mean():.2%} | Validasi n={m_v.sum():,}, rate={yV_c.mean():.2%}")

ta, tv = rate_table(yA_c, FA.loc[m_a, "products_end"]), rate_table(yV_c, FV.loc[m_v, "products_end"])
rate_panel(ta, tv, yA_c.mean(), yV_c.mean(), "Card holders only: tagging rate by number of products", "Products", "10a_cardholders_products.png", fig_dir=FIG); plt.show()
ta, tv = rate_table(yA_c, FA.loc[m_a, "balance_tier"]), rate_table(yV_c, FV.loc[m_v, "balance_tier"])
rate_panel(ta, tv, yA_c.mean(), yV_c.mean(), "Card holders only: tagging rate by balance tier", "Balance tier", "10b_cardholders_balance.png", fig_dir=FIG); plt.show()

cmp_cols = ["n_zero_balance_quarters", "products_mean", "balance_mean", "active_change", "active_rate", "age"]
sp_cmp = pd.DataFrame({
    "All_full": [spearmanr(FA[c], yA)[0] for c in cmp_cols], "All_cardholders": [spearmanr(FA.loc[m_a, c], yA_c)[0] for c in cmp_cols],
    "Val_full": [spearmanr(FV[c], yV)[0] for c in cmp_cols], "Val_cardholders": [spearmanr(FV.loc[m_v, c], yV_c)[0] for c in cmp_cols]}, index=cmp_cols)
display(sp_cmp.round(3))

## H. Why do labels differ between the two windows?
2,042 customers have a different tag in Data All vs Data Validasi. We check whether the customers who *changed* look different from those who did not (on observed features only).

In [ ]:
d = pd.DataFrame({"all": yA, "val": yV})
d["transition"] = d["all"].astype(str) + "->" + d["val"].astype(str)     # e.g. 0->1 = non-tagged in All, tagged in Validasi
print(d["transition"].value_counts().rename("customers").to_frame().assign(share=lambda x: x["customers"] / len(d)).round(3))

prof = FA[["card_start", "products_end", "n_zero_balance_quarters", "active_rate", "balance_mean", "age"]].join(d["transition"])
display(prof.groupby("transition").mean().round(3))

q45 = clean_val.set_index(ID_COL).join(d["transition"])
q45["products_up_q4q5"] = q45["num_products_q5"] > q45["num_products_q4"]
q45["active_down_q4q5"] = q45["active_member_q5"] < q45["active_member_q4"]
q45["balance_up_q4q5"] = q45["balance_q5"] > q45["balance_q4"]
print("Share of customers with a Q4->Q5 change, by label transition:")
display(q45.groupby("transition")[["products_up_q4q5", "active_down_q4q5", "balance_up_q4q5"]].mean().round(3))

## Findings
Summarise in `reports/eda_findings.md` after reviewing the outputs above.